# Advanced YAML & PyYAML — Problems with Complete Solutions

This notebook is a practice-first advanced guide to YAML and PyYAML.

It covers:

- safe parsing and serialization
- scalar type resolution
- nested mappings and sequences
- multi-document YAML
- anchors, aliases, and merge keys
- validation and normalization
- duplicate-key detection
- custom safe tags
- dataclasses
- configuration layering
- environment-variable interpolation
- nested environment overrides
- deterministic output
- secret redaction
- structural diffs
- dependency graphs
- atomic file writes
- production-style configuration pipelines

> **Security baseline:** Prefer `yaml.safe_load`, `yaml.safe_load_all`, `yaml.safe_dump`, and restricted custom loaders derived from `yaml.SafeLoader`. Do not deserialize arbitrary Python objects from untrusted YAML.


## 0. Setup

Install PyYAML if needed:

```bash
python -m pip install pyyaml
```

The exercises use the Python standard library plus PyYAML.


In [ ]:
import os
import re
import tempfile
from copy import deepcopy
from dataclasses import dataclass, asdict, fields, is_dataclass
from datetime import date
from pathlib import Path
from pprint import pprint
from types import MappingProxyType
from typing import Any

import yaml

print("PyYAML version:", yaml.__version__)


## 1. Safe Loader Baseline

Modern PyYAML code should be explicit about loaders.

Recommended defaults:

```python
yaml.safe_load(text)
yaml.safe_load_all(text)
yaml.safe_dump(data)
yaml.safe_dump_all(documents)
```

Parsing YAML is only the first stage. A robust application usually follows:

```text
YAML text
    -> restricted parser
    -> plain Python data
    -> normalization
    -> validation
    -> domain objects
```


In [ ]:
baseline_yaml = """
app:
  name: analytics-api
  debug: false
  retries: 3
  timeout: 2.5
  tags:
    - api
    - internal
"""

baseline = yaml.safe_load(baseline_yaml)
pprint(baseline)

assert baseline["app"]["debug"] is False
assert baseline["app"]["retries"] == 3
assert baseline["app"]["timeout"] == 2.5


# Problem 1 — Predict YAML Scalar Types

Parse the YAML and inspect the Python type of every value.

Questions:

1. Which values become `bool`, `int`, `float`, `None`, `date`, or `str`?
2. How do you force a date-looking value to remain a string?
3. Why should configuration authors quote ambiguous scalars?


In [ ]:
problem_1_yaml = """
enabled: true
disabled: false
count: 42
ratio: 0.875
nothing: null
release_date: 2026-08-07
quoted_date: "2026-08-07"
quoted_number: "42"
empty_value:
message: hello
"""

problem_1_data = yaml.safe_load(problem_1_yaml)


### Solution 1

`safe_load` still performs standard YAML scalar resolution. Quote a value when your application requires text even though it looks like another scalar type.


In [ ]:
for key, value in problem_1_data.items():
    print(f"{key:15} -> {value!r:20} {type(value).__name__}")

assert isinstance(problem_1_data["enabled"], bool)
assert isinstance(problem_1_data["count"], int)
assert isinstance(problem_1_data["ratio"], float)
assert problem_1_data["nothing"] is None
assert isinstance(problem_1_data["release_date"], date)
assert isinstance(problem_1_data["quoted_date"], str)
assert isinstance(problem_1_data["quoted_number"], str)
assert problem_1_data["empty_value"] is None


# Problem 2 — Query a Deeply Nested Configuration

Tasks:

1. Extract the production database host.
2. List enabled feature names.
3. Sum replicas across services.
4. Build `{service_name: port}`.


In [ ]:
problem_2_yaml = """
environment: production
database:
  production:
    host: db.prod.example
    port: 5432
  staging:
    host: db.staging.example
    port: 5432
features:
  - name: search
    enabled: true
  - name: recommendations
    enabled: false
  - name: audit
    enabled: true
services:
  - name: api
    port: 8000
    replicas: 4
  - name: worker
    port: 9000
    replicas: 3
  - name: scheduler
    port: 9100
    replicas: 1
"""

cfg = yaml.safe_load(problem_2_yaml)


### Solution 2


In [ ]:
prod_host = cfg["database"]["production"]["host"]

enabled_features = [
    item["name"]
    for item in cfg["features"]
    if item["enabled"]
]

total_replicas = sum(
    service["replicas"]
    for service in cfg["services"]
)

service_ports = {
    service["name"]: service["port"]
    for service in cfg["services"]
}

print("Production host:", prod_host)
print("Enabled features:", enabled_features)
print("Total replicas:", total_replicas)
print("Service ports:", service_ports)

assert prod_host == "db.prod.example"
assert enabled_features == ["search", "audit"]
assert total_replicas == 8
assert service_ports == {
    "api": 8000,
    "worker": 9000,
    "scheduler": 9100,
}


# Problem 3 — Serialize Python Data Cleanly

Emit readable YAML that:

- uses block style
- preserves insertion order
- emits Unicode directly
- avoids Python-specific tags
- round-trips with `safe_load`


In [ ]:
problem_3_data = {
    "service": "payments",
    "owner": "Miyuki 山田",
    "enabled": True,
    "limits": {
        "requests_per_minute": 1200,
        "burst": 100,
    },
    "regions": ["eu-central-1", "us-east-1"],
}


### Solution 3


In [ ]:
problem_3_yaml = yaml.safe_dump(
    problem_3_data,
    default_flow_style=False,
    sort_keys=False,
    allow_unicode=True,
)

print(problem_3_yaml)

round_trip = yaml.safe_load(problem_3_yaml)
assert round_trip == problem_3_data


# Problem 4 — Multiple YAML Documents

Tasks:

1. Safely parse every document.
2. Keep only enabled documents.
3. Extract their names.
4. Serialize them back as a multi-document YAML stream.


In [ ]:
problem_4_yaml = """
---
name: api
enabled: true
replicas: 3
---
name: worker
enabled: false
replicas: 2
---
name: scheduler
enabled: true
replicas: 1
"""


### Solution 4


In [ ]:
documents = list(yaml.safe_load_all(problem_4_yaml))
enabled_documents = [
    doc for doc in documents
    if doc["enabled"]
]
enabled_names = [
    doc["name"]
    for doc in enabled_documents
]

serialized = yaml.safe_dump_all(
    enabled_documents,
    explicit_start=True,
    sort_keys=False,
)

pprint(documents)
print("Enabled:", enabled_names)
print(serialized)

assert enabled_names == ["api", "scheduler"]


# Problem 5 — Anchors, Aliases, and Merge Keys

Confirm that shared defaults are inherited and that explicit values override merged defaults.


In [ ]:
problem_5_yaml = """
defaults: &defaults
  image: myapp:2.1
  replicas: 2
  restart_policy: always

staging:
  <<: *defaults
  environment: staging

production:
  <<: *defaults
  environment: production
  replicas: 6
"""

problem_5_data = yaml.safe_load(problem_5_yaml)
pprint(problem_5_data)


### Solution 5

Anchors define reusable nodes, aliases reference them, and the merge key combines mappings. They reduce duplication but should not be overused because heavy indirection hurts readability.


In [ ]:
assert problem_5_data["staging"]["image"] == "myapp:2.1"
assert problem_5_data["staging"]["replicas"] == 2
assert problem_5_data["production"]["replicas"] == 6
assert problem_5_data["production"]["restart_policy"] == "always"


# Problem 6 — Graceful Syntax Error Reporting

Write a parser that returns `(data, error_message)` and includes line and column information when PyYAML provides it.


In [ ]:
bad_yaml = """
service:
  name: api
  ports:
    - 8000
    - 8001
   replicas: 3
"""


### Solution 6


In [ ]:
def parse_yaml_safely(text: str):
    try:
        return yaml.safe_load(text), None
    except yaml.YAMLError as exc:
        mark = getattr(exc, "problem_mark", None)

        if mark is not None:
            message = (
                f"YAML error at line {mark.line + 1}, "
                f"column {mark.column + 1}: {exc}"
            )
        else:
            message = f"YAML error: {exc}"

        return None, message


data, error = parse_yaml_safely(bad_yaml)
print("data:", data)
print("error:", error)

assert data is None
assert error is not None


# Problem 7 — Validate a Service Configuration

Contract:

- root must be a mapping
- `name`: required non-empty string
- `port`: integer from 1 to 65535
- `debug`: optional boolean, default `False`
- `workers`: optional positive integer, default `1`

Return normalized data or raise `ValueError`.


In [ ]:
problem_7_yaml = """
name: inventory-api
port: 8080
workers: 4
"""


### Solution 7


In [ ]:
def validate_service_config(raw: Any) -> dict:
    if not isinstance(raw, dict):
        raise ValueError("Configuration must be a mapping")

    name = raw.get("name")
    port = raw.get("port")
    debug = raw.get("debug", False)
    workers = raw.get("workers", 1)

    if not isinstance(name, str) or not name.strip():
        raise ValueError("'name' must be a non-empty string")

    if isinstance(port, bool) or not isinstance(port, int):
        raise ValueError("'port' must be an integer")

    if not 1 <= port <= 65535:
        raise ValueError("'port' must be between 1 and 65535")

    if not isinstance(debug, bool):
        raise ValueError("'debug' must be a boolean")

    if (
        isinstance(workers, bool)
        or not isinstance(workers, int)
        or workers <= 0
    ):
        raise ValueError("'workers' must be a positive integer")

    return {
        "name": name.strip(),
        "port": port,
        "debug": debug,
        "workers": workers,
    }


validated = validate_service_config(
    yaml.safe_load(problem_7_yaml)
)

pprint(validated)

assert validated == {
    "name": "inventory-api",
    "port": 8080,
    "debug": False,
    "workers": 4,
}


In [ ]:
invalid_examples = [
    "name: ''\nport: 8080",
    "name: api\nport: 70000",
    "name: api\nport: true",
    "name: api\nport: 8000\nworkers: 0",
]

for text in invalid_examples:
    try:
        validate_service_config(yaml.safe_load(text))
    except ValueError as exc:
        print("Rejected:", exc)


# Problem 8 — Convert Validated Data to a Dataclass

Keep YAML parsing separate from domain object construction.


### Solution 8


In [ ]:
@dataclass(frozen=True)
class ServiceConfig:
    name: str
    port: int
    debug: bool = False
    workers: int = 1


service = ServiceConfig(**validated)
print(service)

assert service.port == 8080


In [ ]:
service_yaml = yaml.safe_dump(
    asdict(service),
    sort_keys=False,
)

print(service_yaml)
assert yaml.safe_load(service_yaml)["name"] == "inventory-api"


# Problem 9 — Recursive Configuration Layering

Implement a deep merge where:

- nested mappings merge recursively
- non-mapping override values replace base values
- neither input is mutated


In [ ]:
base_yaml = """
app:
  name: catalog
  logging:
    level: INFO
    json: false
  database:
    host: localhost
    port: 5432
    pool:
      min: 2
      max: 10
"""

prod_yaml = """
app:
  logging:
    level: WARNING
    json: true
  database:
    host: db.prod.internal
    pool:
      max: 40
"""


### Solution 9


In [ ]:
def deep_merge(base: dict, override: dict) -> dict:
    result = deepcopy(base)

    for key, value in override.items():
        if (
            key in result
            and isinstance(result[key], dict)
            and isinstance(value, dict)
        ):
            result[key] = deep_merge(result[key], value)
        else:
            result[key] = deepcopy(value)

    return result


base = yaml.safe_load(base_yaml)
prod = yaml.safe_load(prod_yaml)
merged = deep_merge(base, prod)

pprint(merged)

assert merged["app"]["name"] == "catalog"
assert merged["app"]["logging"] == {
    "level": "WARNING",
    "json": True,
}
assert merged["app"]["database"]["port"] == 5432
assert merged["app"]["database"]["pool"] == {
    "min": 2,
    "max": 40,
}


# Problem 10 — Environment Variable Interpolation

Support:

- `${NAME}`
- `${NAME:-default}`

Process recursively after YAML parsing.


In [ ]:
problem_10_yaml = """
database:
  host: "${DB_HOST:-localhost}"
  port: 5432
  username: "${DB_USER}"
workers:
  - queue: email
    token: "${EMAIL_TOKEN:-development-token}"
  - queue: billing
    token: "${BILLING_TOKEN:-development-token}"
"""


### Solution 10


In [ ]:
ENV_PATTERN = re.compile(
    r"\$\{([A-Za-z_][A-Za-z0-9_]*)(?::-([^}]*))?\}"
)


def expand_env_string(value: str, env: dict[str, str]) -> str:
    def replace(match: re.Match) -> str:
        name = match.group(1)
        default = match.group(2)

        if name in env:
            return env[name]

        if default is not None:
            return default

        raise KeyError(
            f"Missing required environment variable: {name}"
        )

    return ENV_PATTERN.sub(replace, value)


def interpolate_env(value: Any, env: dict[str, str]) -> Any:
    if isinstance(value, str):
        return expand_env_string(value, env)

    if isinstance(value, list):
        return [
            interpolate_env(item, env)
            for item in value
        ]

    if isinstance(value, dict):
        return {
            key: interpolate_env(item, env)
            for key, item in value.items()
        }

    return value


raw = yaml.safe_load(problem_10_yaml)

expanded = interpolate_env(
    raw,
    {
        "DB_HOST": "db.internal",
        "DB_USER": "app_user",
    },
)

pprint(expanded)

assert expanded["database"]["host"] == "db.internal"
assert expanded["database"]["username"] == "app_user"
assert expanded["workers"][0]["token"] == "development-token"


# Problem 11 — Parse Environment Override Values as Native Types

Environment variables are strings, but configuration values may be booleans, integers, lists, or mappings.


### Solution 11


In [ ]:
def parse_env_value(text: str) -> Any:
    return yaml.safe_load(text)


examples = [
    "42",
    "false",
    "3.14",
    "[a, b]",
    "{x: 1, y: 2}",
    "hello",
]

for item in examples:
    parsed = parse_env_value(item)
    print(
        f"{item!r:20} -> "
        f"{parsed!r:25} "
        f"({type(parsed).__name__})"
    )

assert parse_env_value("42") == 42
assert parse_env_value("false") is False
assert parse_env_value("[a, b]") == ["a", "b"]


# Problem 12 — Nested Overrides from Environment Variable Names

Interpret double underscores as path separators.

Example:

```text
APP__LOGGING__LEVEL=DEBUG
APP__DATABASE__POOL__MAX=100
```


### Solution 12


In [ ]:
def set_nested(
    config: dict,
    path: list[str],
    value: Any,
) -> None:
    current = config

    for key in path[:-1]:
        if (
            key not in current
            or not isinstance(current[key], dict)
        ):
            current[key] = {}

        current = current[key]

    current[path[-1]] = value


config = yaml.safe_load(base_yaml)

overrides = {
    "APP__LOGGING__LEVEL": "DEBUG",
    "APP__DATABASE__POOL__MAX": "100",
    "APP__LOGGING__JSON": "true",
}

for env_name, env_value in overrides.items():
    path = [
        part.lower()
        for part in env_name.split("__")
    ]
    set_nested(
        config,
        path,
        parse_env_value(env_value),
    )

pprint(config)

assert config["app"]["logging"]["level"] == "DEBUG"
assert config["app"]["logging"]["json"] is True
assert config["app"]["database"]["pool"]["max"] == 100


# Problem 13 — Reject Duplicate Mapping Keys

Duplicate keys can silently hide configuration mistakes.

Create a loader derived from `SafeLoader` that rejects duplicates.


### Solution 13


In [ ]:
class UniqueKeyLoader(yaml.SafeLoader):
    pass


def construct_mapping_no_duplicates(
    loader,
    node,
    deep=False,
):
    mapping = {}

    for key_node, value_node in node.value:
        key = loader.construct_object(
            key_node,
            deep=deep,
        )

        if key in mapping:
            raise yaml.constructor.ConstructorError(
                "while constructing a mapping",
                node.start_mark,
                f"found duplicate key: {key!r}",
                key_node.start_mark,
            )

        value = loader.construct_object(
            value_node,
            deep=deep,
        )
        mapping[key] = value

    return mapping


UniqueKeyLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG,
    construct_mapping_no_duplicates,
)


duplicate_yaml = """
database:
  host: db1
  host: db2
"""

try:
    yaml.load(
        duplicate_yaml,
        Loader=UniqueKeyLoader,
    )
except yaml.constructor.ConstructorError as exc:
    print("Duplicate rejected:")
    print(exc)


# Problem 14 — Custom Safe `!duration` Tag

Support:

- `250ms`
- `3s`
- `5m`
- `1.5h`

Convert to seconds as a float. Reject malformed values.


### Solution 14


In [ ]:
DURATION_RE = re.compile(
    r"^(?P<number>\d+(?:\.\d+)?)(?P<unit>ms|s|m|h)$"
)

DURATION_FACTORS = {
    "ms": 0.001,
    "s": 1.0,
    "m": 60.0,
    "h": 3600.0,
}


class ConfigLoader(yaml.SafeLoader):
    pass


def construct_duration(loader, node):
    text = loader.construct_scalar(node)
    match = DURATION_RE.fullmatch(text)

    if not match:
        raise yaml.constructor.ConstructorError(
            None,
            None,
            f"invalid !duration value: {text!r}",
            node.start_mark,
        )

    number = float(match.group("number"))
    unit = match.group("unit")

    return number * DURATION_FACTORS[unit]


ConfigLoader.add_constructor(
    "!duration",
    construct_duration,
)


duration_yaml = """
timeout: !duration 250ms
retry_delay: !duration 3s
cache_ttl: !duration 5m
job_timeout: !duration 1.5h
"""

durations = yaml.load(
    duration_yaml,
    Loader=ConfigLoader,
)

pprint(durations)

assert durations["timeout"] == 0.25
assert durations["retry_delay"] == 3.0
assert durations["cache_ttl"] == 300.0
assert durations["job_timeout"] == 5400.0


In [ ]:
invalid_duration_yaml = """
timeout: !duration tomorrow
"""

try:
    yaml.load(
        invalid_duration_yaml,
        Loader=ConfigLoader,
    )
except yaml.constructor.ConstructorError as exc:
    print("Invalid duration rejected:")
    print(exc)


# Problem 15 — Custom Representer for a Domain Type

Create a `Duration` dataclass and safely serialize it as `!duration` without enabling arbitrary Python tags.


### Solution 15


In [ ]:
@dataclass(frozen=True)
class Duration:
    seconds: float


class ConfigDumper(yaml.SafeDumper):
    pass


def represent_duration(
    dumper,
    value: Duration,
):
    scalar = f"{value.seconds:g}s"
    return dumper.represent_scalar(
        "!duration",
        scalar,
    )


ConfigDumper.add_representer(
    Duration,
    represent_duration,
)


duration_objects = {
    "timeout": Duration(2.5),
    "retry_delay": Duration(10),
}

duration_text = yaml.dump(
    duration_objects,
    Dumper=ConfigDumper,
    sort_keys=False,
)

print(duration_text)


# Problem 16 — Controlled Custom-Type Round Trip

Make a safe loader that reconstructs `Duration` objects and prove round-trip equality.


### Solution 16


In [ ]:
class DurationLoader(yaml.SafeLoader):
    pass


def construct_duration_object(
    loader,
    node,
):
    text = loader.construct_scalar(node)
    match = DURATION_RE.fullmatch(text)

    if not match:
        raise yaml.constructor.ConstructorError(
            None,
            None,
            f"invalid !duration value: {text!r}",
            node.start_mark,
        )

    number = float(match.group("number"))
    unit = match.group("unit")

    return Duration(
        number * DURATION_FACTORS[unit]
    )


DurationLoader.add_constructor(
    "!duration",
    construct_duration_object,
)


original = {
    "timeout": Duration(2.5),
    "retry_delay": Duration(10),
}

serialized = yaml.dump(
    original,
    Dumper=ConfigDumper,
    sort_keys=False,
)

restored = yaml.load(
    serialized,
    Loader=DurationLoader,
)

print(serialized)
print(restored)

assert restored == original


# Problem 17 — Deterministic YAML for Version Control

Generate stable YAML:

- block style
- sorted keys
- Unicode
- no aliases


### Solution 17


In [ ]:
class NoAliasSafeDumper(yaml.SafeDumper):
    def ignore_aliases(self, data):
        return True


def deterministic_yaml(data: Any) -> str:
    return yaml.dump(
        data,
        Dumper=NoAliasSafeDumper,
        default_flow_style=False,
        sort_keys=True,
        allow_unicode=True,
    )


shared = {
    "retries": 3,
    "timeout": 5,
}

data = {
    "worker": shared,
    "api": shared,
}

print(deterministic_yaml(data))


# Problem 18 — Deep-Freeze Configuration

Convert:

- dictionaries to `MappingProxyType`
- lists to tuples
- scalars unchanged


### Solution 18


In [ ]:
def deep_freeze(value: Any) -> Any:
    if isinstance(value, dict):
        return MappingProxyType({
            key: deep_freeze(item)
            for key, item in value.items()
        })

    if isinstance(value, list):
        return tuple(
            deep_freeze(item)
            for item in value
        )

    return value


frozen = deep_freeze(
    yaml.safe_load(problem_2_yaml)
)

print(frozen["services"][0]["name"])

try:
    frozen["environment"] = "dev"
except TypeError as exc:
    print("Mutation blocked:", exc)


# Problem 19 — Flatten and Unflatten Nested Mappings

Convert nested keys to dotted paths and back.


### Solution 19


In [ ]:
def flatten_mapping(
    data: dict,
    prefix: str = "",
) -> dict:
    result = {}

    for key, value in data.items():
        full_key = (
            f"{prefix}.{key}"
            if prefix
            else str(key)
        )

        if isinstance(value, dict):
            result.update(
                flatten_mapping(
                    value,
                    full_key,
                )
            )
        else:
            result[full_key] = value

    return result


def unflatten_mapping(data: dict) -> dict:
    result = {}

    for dotted_key, value in data.items():
        parts = dotted_key.split(".")
        current = result

        for part in parts[:-1]:
            current = current.setdefault(
                part,
                {},
            )

        current[parts[-1]] = value

    return result


nested = {
    "database": {
        "host": "localhost",
        "pool": {"max": 20},
    },
    "debug": False,
}

flat = flatten_mapping(nested)
restored = unflatten_mapping(flat)

pprint(flat)
pprint(restored)

assert restored == nested


# Problem 20 — Structural Configuration Diff

Report:

- added keys
- removed keys
- changed values


### Solution 20


In [ ]:
def config_diff(
    old: dict,
    new: dict,
) -> dict:
    old_flat = flatten_mapping(old)
    new_flat = flatten_mapping(new)

    old_keys = set(old_flat)
    new_keys = set(new_flat)

    added = {
        key: new_flat[key]
        for key in sorted(
            new_keys - old_keys
        )
    }

    removed = {
        key: old_flat[key]
        for key in sorted(
            old_keys - new_keys
        )
    }

    changed = {
        key: {
            "old": old_flat[key],
            "new": new_flat[key],
        }
        for key in sorted(
            old_keys & new_keys
        )
        if old_flat[key] != new_flat[key]
    }

    return {
        "added": added,
        "removed": removed,
        "changed": changed,
    }


old_cfg = yaml.safe_load("""
app:
  workers: 2
  debug: false
database:
  host: localhost
""")

new_cfg = yaml.safe_load("""
app:
  workers: 6
  feature_x: true
database:
  host: db.prod
""")

diff = config_diff(old_cfg, new_cfg)
pprint(diff)

assert diff["added"]["app.feature_x"] is True
assert diff["changed"]["app.workers"] == {
    "old": 2,
    "new": 6,
}


# Problem 21 — Redact Secrets Before Logging

Mask values whose key contains:

- password
- secret
- token
- api_key
- private_key


### Solution 21


In [ ]:
SENSITIVE_KEY_PARTS = {
    "password",
    "secret",
    "token",
    "api_key",
    "private_key",
}


def is_sensitive_key(key: Any) -> bool:
    normalized = str(key).lower()

    return any(
        part in normalized
        for part in SENSITIVE_KEY_PARTS
    )


def redact_secrets(value: Any) -> Any:
    if isinstance(value, dict):
        result = {}

        for key, item in value.items():
            if is_sensitive_key(key):
                result[key] = "***REDACTED***"
            else:
                result[key] = redact_secrets(
                    item
                )

        return result

    if isinstance(value, list):
        return [
            redact_secrets(item)
            for item in value
        ]

    return value


secret_yaml = """
database:
  username: app
  password: super-secret
integrations:
  github:
    api_token: ghp_example
  service:
    api_key: abc123
"""

secret_cfg = yaml.safe_load(secret_yaml)
safe_for_logs = redact_secrets(secret_cfg)

print(
    yaml.safe_dump(
        safe_for_logs,
        sort_keys=False,
    )
)

assert safe_for_logs["database"]["password"] == "***REDACTED***"
assert safe_for_logs["integrations"]["github"]["api_token"] == "***REDACTED***"


# Problem 22 — Reject Unexpected Keys

Reject unknown keys so typos such as `workres` do not silently pass.


### Solution 22


In [ ]:
ALLOWED_SERVICE_KEYS = {
    "name",
    "port",
    "debug",
    "workers",
}


def reject_unknown_keys(
    raw: dict,
    allowed: set[str],
) -> None:
    unknown = set(raw) - allowed

    if unknown:
        raise ValueError(
            "Unknown configuration key(s): "
            + ", ".join(sorted(unknown))
        )


typo_yaml = """
name: api
port: 8080
workres: 4
"""

try:
    raw = yaml.safe_load(typo_yaml)
    reject_unknown_keys(
        raw,
        ALLOWED_SERVICE_KEYS,
    )
except ValueError as exc:
    print("Rejected:", exc)


# Problem 23 — Production-Style Loader Pipeline

Build one function that performs:

1. strict duplicate-key parsing
2. environment interpolation
3. unknown-key rejection
4. business validation
5. dataclass construction


### Solution 23


In [ ]:
class StrictSafeLoader(yaml.SafeLoader):
    pass


StrictSafeLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG,
    construct_mapping_no_duplicates,
)


def load_service_config(
    text: str,
    env: dict[str, str] | None = None,
) -> ServiceConfig:
    env = {} if env is None else env

    try:
        raw = yaml.load(
            text,
            Loader=StrictSafeLoader,
        )
    except yaml.YAMLError as exc:
        raise ValueError(
            f"Invalid YAML: {exc}"
        ) from exc

    raw = interpolate_env(raw, env)

    if not isinstance(raw, dict):
        raise ValueError(
            "Configuration must be a mapping"
        )

    reject_unknown_keys(
        raw,
        ALLOWED_SERVICE_KEYS,
    )

    normalized = validate_service_config(
        raw
    )

    return ServiceConfig(**normalized)


production_style_yaml = """
name: "${SERVICE_NAME:-orders}"
port: 8080
debug: false
workers: 8
"""

service_cfg = load_service_config(
    production_style_yaml,
    env={
        "SERVICE_NAME": "orders-api",
    },
)

print(service_cfg)

assert service_cfg == ServiceConfig(
    name="orders-api",
    port=8080,
    debug=False,
    workers=8,
)


# Problem 24 — Cross-Record Validation

Validate a list of services.

Rules:

- every service passes the service validator
- names are unique
- ports are unique


In [ ]:
problem_24_yaml = """
services:
  - name: api
    port: 8000
    workers: 4
  - name: admin
    port: 8001
    workers: 2
  - name: metrics
    port: 9090
"""


### Solution 24


In [ ]:
def validate_service_list(
    raw: Any,
) -> list[ServiceConfig]:
    if not isinstance(raw, dict):
        raise ValueError(
            "Root must be a mapping"
        )

    services = raw.get("services")

    if not isinstance(services, list):
        raise ValueError(
            "'services' must be a list"
        )

    result = []
    names = set()
    ports = set()

    for index, item in enumerate(
        services
    ):
        if not isinstance(item, dict):
            raise ValueError(
                f"Service {index} must be a mapping"
            )

        reject_unknown_keys(
            item,
            ALLOWED_SERVICE_KEYS,
        )

        normalized = validate_service_config(
            item
        )

        if normalized["name"] in names:
            raise ValueError(
                "Duplicate service name: "
                f"{normalized['name']}"
            )

        if normalized["port"] in ports:
            raise ValueError(
                "Duplicate service port: "
                f"{normalized['port']}"
            )

        names.add(normalized["name"])
        ports.add(normalized["port"])
        result.append(
            ServiceConfig(**normalized)
        )

    return result


services = validate_service_list(
    yaml.safe_load(problem_24_yaml)
)

pprint(services)

assert len(services) == 3
assert services[0].workers == 4


# Problem 25 — YAML Routing Table

Each route contains:

- path
- method
- handler
- optional `auth`, default `true`

Normalize methods to uppercase, enforce supported methods, and reject duplicate `(method, path)` routes.


In [ ]:
routing_yaml = """
routes:
  - path: /health
    method: get
    handler: health_check
    auth: false

  - path: /users
    method: GET
    handler: list_users

  - path: /users
    method: POST
    handler: create_user
"""


### Solution 25


In [ ]:
ALLOWED_METHODS = {
    "GET",
    "POST",
    "PUT",
    "PATCH",
    "DELETE",
}


def build_routing_table(
    raw: dict,
) -> dict:
    routes = raw.get("routes")

    if not isinstance(routes, list):
        raise ValueError(
            "'routes' must be a list"
        )

    table = {}

    for index, route in enumerate(routes):
        if not isinstance(route, dict):
            raise ValueError(
                f"Route {index} must be a mapping"
            )

        path = route.get("path")
        method = route.get("method")
        handler = route.get("handler")
        auth = route.get("auth", True)

        if (
            not isinstance(path, str)
            or not path.startswith("/")
        ):
            raise ValueError(
                f"Route {index}: invalid path"
            )

        if not isinstance(method, str):
            raise ValueError(
                f"Route {index}: invalid method"
            )

        method = method.upper()

        if method not in ALLOWED_METHODS:
            raise ValueError(
                f"Route {index}: unsupported "
                f"method {method}"
            )

        if (
            not isinstance(handler, str)
            or not handler
        ):
            raise ValueError(
                f"Route {index}: invalid handler"
            )

        if not isinstance(auth, bool):
            raise ValueError(
                f"Route {index}: auth "
                "must be boolean"
            )

        key = (method, path)

        if key in table:
            raise ValueError(
                f"Duplicate route: "
                f"{method} {path}"
            )

        table[key] = {
            "handler": handler,
            "auth": auth,
        }

    return table


routing_table = build_routing_table(
    yaml.safe_load(routing_yaml)
)

pprint(routing_table)

assert routing_table[
    ("GET", "/health")
]["auth"] is False

assert routing_table[
    ("POST", "/users")
]["handler"] == "create_user"


# Problem 26 — CI/CD Dependency Graph

Validate a YAML pipeline.

Rules:

- `needs` defaults to an empty list
- each dependency must exist
- no self-dependency
- no dependency cycle
- return a valid execution order


In [ ]:
pipeline_yaml = """
jobs:
  lint:
    command: ruff check .
  test:
    command: pytest -q
    needs: [lint]
  package:
    command: python -m build
    needs: [test]
  deploy:
    command: ./deploy.sh
    needs:
      - package
"""


### Solution 26


In [ ]:
def validate_pipeline(
    raw: dict,
) -> list[str]:
    jobs = raw.get("jobs")

    if (
        not isinstance(jobs, dict)
        or not jobs
    ):
        raise ValueError(
            "'jobs' must be a non-empty mapping"
        )

    graph = {}

    for name, spec in jobs.items():
        if not isinstance(spec, dict):
            raise ValueError(
                f"Job {name!r} must be a mapping"
            )

        needs = spec.get("needs", [])

        if (
            not isinstance(needs, list)
            or not all(
                isinstance(dep, str)
                for dep in needs
            )
        ):
            raise ValueError(
                f"Job {name!r}: 'needs' must "
                "be a list of strings"
            )

        for dep in needs:
            if dep not in jobs:
                raise ValueError(
                    f"Job {name!r} depends on "
                    f"unknown job {dep!r}"
                )

            if dep == name:
                raise ValueError(
                    f"Job {name!r} cannot "
                    "depend on itself"
                )

        graph[name] = needs

    temporary = set()
    permanent = set()
    order = []

    def visit(node: str):
        if node in permanent:
            return

        if node in temporary:
            raise ValueError(
                "Dependency cycle detected "
                f"at job {node!r}"
            )

        temporary.add(node)

        for dep in graph[node]:
            visit(dep)

        temporary.remove(node)
        permanent.add(node)
        order.append(node)

    for name in jobs:
        visit(name)

    return order


pipeline = yaml.safe_load(
    pipeline_yaml
)

execution_order = validate_pipeline(
    pipeline
)

print(
    "Execution order:",
    execution_order,
)

assert (
    execution_order.index("lint")
    < execution_order.index("test")
    < execution_order.index("package")
    < execution_order.index("deploy")
)


In [ ]:
cyclic_pipeline_yaml = """
jobs:
  a:
    needs: [c]
  b:
    needs: [a]
  c:
    needs: [b]
"""

try:
    validate_pipeline(
        yaml.safe_load(
            cyclic_pipeline_yaml
        )
    )
except ValueError as exc:
    print(
        "Cycle correctly rejected:",
        exc,
    )


# Problem 27 — Safe YAML File Loading

Create a helper that:

- uses UTF-8
- distinguishes file errors from YAML errors
- uses the strict safe loader


### Solution 27


In [ ]:
def load_yaml_file(
    path: str | Path,
) -> Any:
    path = Path(path)

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except OSError as exc:
        raise RuntimeError(
            f"Could not read {path}: {exc}"
        ) from exc

    try:
        return yaml.load(
            text,
            Loader=StrictSafeLoader,
        )
    except yaml.YAMLError as exc:
        raise ValueError(
            f"Invalid YAML in {path}: {exc}"
        ) from exc


with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "config.yaml"

    path.write_text(
        "name: demo\nport: 8080\n",
        encoding="utf-8",
    )

    loaded = load_yaml_file(path)
    print(loaded)

    assert loaded == {
        "name": "demo",
        "port": 8080,
    }


# Problem 28 — Atomic YAML Writes

Serialize first, write to a temporary file in the same directory, flush, and replace the destination with `os.replace`.


### Solution 28


In [ ]:
def atomic_write_yaml(
    path: str | Path,
    data: Any,
) -> None:
    path = Path(path)

    text = yaml.safe_dump(
        data,
        sort_keys=False,
        allow_unicode=True,
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fd, temp_name = tempfile.mkstemp(
        dir=path.parent,
        prefix=f".{path.name}.",
        suffix=".tmp",
        text=True,
    )

    try:
        with os.fdopen(
            fd,
            "w",
            encoding="utf-8",
        ) as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())

        os.replace(
            temp_name,
            path,
        )

    except Exception:
        try:
            os.unlink(temp_name)
        except FileNotFoundError:
            pass
        raise


with tempfile.TemporaryDirectory() as tmp:
    target = Path(tmp) / "config.yaml"

    atomic_write_yaml(
        target,
        {
            "name": "atomic-demo",
            "enabled": True,
        },
    )

    print(
        target.read_text(
            encoding="utf-8"
        )
    )


# Problem 29 — Multiline Strings

Compare YAML literal (`|`) and folded (`>`) block scalars.


In [ ]:
multiline_yaml = """
literal: |
  line one
  line two
  line three

folded: >
  line one
  line two
  line three
"""

multiline = yaml.safe_load(
    multiline_yaml
)

print(
    "literal repr:",
    repr(multiline["literal"]),
)

print(
    "folded repr:",
    repr(multiline["folded"]),
)


### Solution 29


In [ ]:
assert multiline["literal"] == (
    "line one\n"
    "line two\n"
    "line three\n"
)

assert multiline["folded"] == (
    "line one line two line three\n"
)

print(
    yaml.safe_dump(
        multiline,
        sort_keys=False,
    )
)


# Problem 30 — Prove Deep Merge Does Not Mutate Inputs

Use copies and assertions to verify immutability of source dictionaries.


### Solution 30


In [ ]:
base = {
    "database": {
        "pool": {
            "min": 2,
            "max": 10,
        }
    }
}

override = {
    "database": {
        "pool": {
            "max": 50,
        }
    }
}

base_before = deepcopy(base)
override_before = deepcopy(override)

merged = deep_merge(
    base,
    override,
)

assert base == base_before
assert override == override_before

assert merged["database"]["pool"] == {
    "min": 2,
    "max": 50,
}

print("Inputs unchanged.")


# Problem 31 — Defensive Input Limits

`safe_load` blocks arbitrary Python object construction, but applications may also want size and nesting limits for untrusted configuration.


### Solution 31


In [ ]:
def structural_depth(
    value: Any,
) -> int:
    if isinstance(value, dict):
        return 1 + max(
            (
                structural_depth(item)
                for item in value.values()
            ),
            default=0,
        )

    if isinstance(value, list):
        return 1 + max(
            (
                structural_depth(item)
                for item in value
            ),
            default=0,
        )

    return 0


def bounded_safe_load(
    text: str,
    *,
    max_chars: int = 100_000,
    max_lines: int = 5_000,
    max_nesting: int = 30,
) -> Any:
    if len(text) > max_chars:
        raise ValueError(
            "YAML input exceeds maximum "
            "character count"
        )

    if text.count("\n") + 1 > max_lines:
        raise ValueError(
            "YAML input exceeds maximum "
            "line count"
        )

    data = yaml.safe_load(text)

    if (
        structural_depth(data)
        > max_nesting
    ):
        raise ValueError(
            "YAML input exceeds maximum "
            "nesting depth"
        )

    return data


small = bounded_safe_load(
    "a:\n  b:\n    c: 1\n"
)

pprint(small)

assert structural_depth(
    {"a": {"b": {"c": 1}}}
) == 3


# Problem 32 — Demonstrate Safe Rejection of Python-Specific Tags

Do not execute dangerous payloads. Use a harmless Python-specific tag and verify `safe_load` rejects it.


### Solution 32


In [ ]:
python_specific_yaml = """
value: !!python/tuple [1, 2, 3]
"""

try:
    yaml.safe_load(
        python_specific_yaml
    )
except yaml.constructor.ConstructorError as exc:
    print(
        "safe_load rejected the "
        "Python-specific tag:"
    )
    print(exc)


# Problem 33 — Custom Safe `!date_range` Tag

Parse:

```yaml
window: !date_range 2026-08-01..2026-08-31
```

Return two `date` objects and reject reversed ranges.


### Solution 33


In [ ]:
DATE_RANGE_RE = re.compile(
    r"^(\d{4}-\d{2}-\d{2})"
    r"\.\."
    r"(\d{4}-\d{2}-\d{2})$"
)


class DateRangeLoader(
    yaml.SafeLoader
):
    pass


def construct_date_range(
    loader,
    node,
):
    text = loader.construct_scalar(node)
    match = DATE_RANGE_RE.fullmatch(
        text
    )

    if not match:
        raise yaml.constructor.ConstructorError(
            None,
            None,
            f"invalid date range: {text!r}",
            node.start_mark,
        )

    start = date.fromisoformat(
        match.group(1)
    )
    end = date.fromisoformat(
        match.group(2)
    )

    if start > end:
        raise yaml.constructor.ConstructorError(
            None,
            None,
            "date range start is after end",
            node.start_mark,
        )

    return start, end


DateRangeLoader.add_constructor(
    "!date_range",
    construct_date_range,
)


range_data = yaml.load(
    """
window: !date_range 2026-08-01..2026-08-31
""",
    Loader=DateRangeLoader,
)

print(range_data)

assert range_data["window"] == (
    date(2026, 8, 1),
    date(2026, 8, 31),
)


# Problem 34 — Normalize Mapping Keys to `snake_case`

Handle keys such as:

- `maxRetries`
- `max-retries`
- `Max Retries`

Reject collisions after normalization.


### Solution 34


In [ ]:
def to_snake_case(
    name: str,
) -> str:
    name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        name,
    )

    name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        name,
    )

    return name.strip("_").lower()


def normalize_keys(
    value: Any,
) -> Any:
    if isinstance(value, list):
        return [
            normalize_keys(item)
            for item in value
        ]

    if isinstance(value, dict):
        result = {}

        for key, item in value.items():
            new_key = (
                to_snake_case(key)
                if isinstance(key, str)
                else key
            )

            if new_key in result:
                raise ValueError(
                    "Key collision after "
                    f"normalization: {new_key!r}"
                )

            result[new_key] = (
                normalize_keys(item)
            )

        return result

    return value


mixed_keys_yaml = """
Service Name: catalog
maxRetries: 5
database-config:
  poolSize: 20
"""

normalized = normalize_keys(
    yaml.safe_load(mixed_keys_yaml)
)

pprint(normalized)

assert normalized == {
    "service_name": "catalog",
    "max_retries": 5,
    "database_config": {
        "pool_size": 20,
    },
}


# Problem 35 — Convert Domain Objects to Plain Data

Create a generic converter for dataclasses, dictionaries, lists, and tuples, then serialize only plain data with `safe_dump`.


### Solution 35


In [ ]:
def to_plain_data(
    value: Any,
) -> Any:
    if is_dataclass(value):
        return {
            field.name: to_plain_data(
                getattr(
                    value,
                    field.name,
                )
            )
            for field in fields(value)
        }

    if isinstance(value, dict):
        return {
            to_plain_data(key): (
                to_plain_data(item)
            )
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            to_plain_data(item)
            for item in value
        ]

    return value


objects = {
    "primary": ServiceConfig(
        name="api",
        port=8000,
        workers=4,
    ),
    "fallback": ServiceConfig(
        name="api-backup",
        port=8001,
        workers=2,
    ),
}

plain = to_plain_data(objects)

print(
    yaml.safe_dump(
        plain,
        sort_keys=False,
    )
)


# Problem 36 — Mini Configuration Framework

Combine:

- strict parsing
- environment interpolation
- deep merge
- environment overrides
- redaction
- deterministic serialization


### Solution 36


In [ ]:
class YamlConfigManager:
    def __init__(
        self,
        defaults: dict | None = None,
    ):
        self.defaults = deepcopy(
            defaults or {}
        )

    def loads(
        self,
        text: str,
        *,
        env: dict[str, str] | None = None,
    ) -> dict:
        env = (
            {}
            if env is None
            else env
        )

        try:
            parsed = yaml.load(
                text,
                Loader=StrictSafeLoader,
            )
        except yaml.YAMLError as exc:
            raise ValueError(
                f"Invalid YAML: {exc}"
            ) from exc

        if parsed is None:
            parsed = {}

        if not isinstance(parsed, dict):
            raise ValueError(
                "Top-level configuration "
                "must be a mapping"
            )

        parsed = interpolate_env(
            parsed,
            env,
        )

        return deep_merge(
            self.defaults,
            parsed,
        )

    def apply_env_overrides(
        self,
        config: dict,
        overrides: dict[str, str],
        *,
        prefix: str = "APP__",
    ) -> dict:
        result = deepcopy(config)

        for name, raw_value in (
            overrides.items()
        ):
            if not name.startswith(
                prefix
            ):
                continue

            suffix = name[len(prefix):]

            if not suffix:
                continue

            path = [
                part.lower()
                for part in suffix.split("__")
                if part
            ]

            if not path:
                continue

            set_nested(
                result,
                path,
                parse_env_value(
                    raw_value
                ),
            )

        return result

    def dump(
        self,
        config: dict,
    ) -> str:
        return deterministic_yaml(
            config
        )

    def safe_log_view(
        self,
        config: dict,
    ) -> dict:
        return redact_secrets(
            config
        )


In [ ]:
manager = YamlConfigManager(
    defaults={
        "logging": {
            "level": "INFO",
            "json": False,
        },
        "workers": 2,
    }
)

app_yaml = """
name: "${APP_NAME:-demo}"
logging:
  json: true
database:
  password: "${DB_PASSWORD}"
"""

loaded = manager.loads(
    app_yaml,
    env={
        "APP_NAME": "billing",
        "DB_PASSWORD": "do-not-log-me",
    },
)

final = manager.apply_env_overrides(
    loaded,
    {
        "APP__WORKERS": "8",
        "APP__LOGGING__LEVEL": "WARNING",
    },
)

print("Runtime configuration:")
pprint(final)

print("\nSafe log view:")
pprint(
    manager.safe_log_view(final)
)

print("\nDeterministic YAML:")
print(manager.dump(final))

assert final["workers"] == 8
assert final["logging"]["level"] == "WARNING"

assert (
    manager.safe_log_view(final)
    ["database"]["password"]
    == "***REDACTED***"
)


# Problem 37 — Test the Configuration Utilities

Create assertions for:

- valid parsing
- duplicate rejection
- invalid port rejection
- missing environment variable rejection
- custom duration parsing
- redaction


### Solution 37


In [ ]:
def run_configuration_tests():
    valid = load_service_config(
        "name: api\nport: 8000\n"
    )

    assert valid.name == "api"

    try:
        load_service_config(
            "name: api\n"
            "name: duplicate\n"
            "port: 8000\n"
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Duplicate key should "
            "have been rejected"
        )

    try:
        load_service_config(
            "name: api\nport: 99999\n"
        )
    except ValueError:
        pass
    else:
        raise AssertionError(
            "Invalid port should "
            "have been rejected"
        )

    try:
        interpolate_env(
            {
                "token": (
                    "${REQUIRED_TOKEN}"
                )
            },
            {},
        )
    except KeyError:
        pass
    else:
        raise AssertionError(
            "Missing environment "
            "variable should fail"
        )

    parsed = yaml.load(
        "delay: !duration 2m",
        Loader=ConfigLoader,
    )

    assert parsed["delay"] == 120.0

    redacted = redact_secrets(
        {"password": "abc"}
    )

    assert redacted["password"] == (
        "***REDACTED***"
    )

    return (
        "All configuration "
        "tests passed."
    )


run_configuration_tests()


# Problem 38 — Kubernetes-Like Resource Index

Parse several YAML documents.

Requirements:

- validate `kind`
- validate `metadata.name`
- validate `spec`
- index by `(kind, name)`
- reject duplicate resources
- find all resources labeled `tier: backend`


In [ ]:
resources_yaml = """
---
kind: Service
metadata:
  name: users
  labels:
    tier: backend
spec:
  port: 8000
---
kind: Deployment
metadata:
  name: users
  labels:
    tier: backend
spec:
  replicas: 4
---
kind: Service
metadata:
  name: frontend
  labels:
    tier: frontend
spec:
  port: 8080
"""


### Solution 38


In [ ]:
def build_resource_index(
    text: str,
):
    index = {}

    for number, resource in enumerate(
        yaml.safe_load_all(text),
        start=1,
    ):
        if not isinstance(
            resource,
            dict,
        ):
            raise ValueError(
                f"Document {number} "
                "must be a mapping"
            )

        kind = resource.get("kind")
        metadata = resource.get(
            "metadata"
        )
        spec = resource.get("spec")

        if (
            not isinstance(kind, str)
            or not kind
        ):
            raise ValueError(
                f"Document {number}: "
                "invalid kind"
            )

        if not isinstance(
            metadata,
            dict,
        ):
            raise ValueError(
                f"Document {number}: "
                "invalid metadata"
            )

        name = metadata.get("name")

        if (
            not isinstance(name, str)
            or not name
        ):
            raise ValueError(
                f"Document {number}: "
                "invalid metadata.name"
            )

        if not isinstance(spec, dict):
            raise ValueError(
                f"Document {number}: "
                "invalid spec"
            )

        key = (kind, name)

        if key in index:
            raise ValueError(
                "Duplicate resource: "
                f"{kind}/{name}"
            )

        index[key] = resource

    return index


resource_index = (
    build_resource_index(
        resources_yaml
    )
)

backend_resources = [
    f"{kind}/{name}"
    for (
        kind,
        name,
    ), resource in (
        resource_index.items()
    )
    if (
        resource
        .get("metadata", {})
        .get("labels", {})
        .get("tier")
        == "backend"
    )
]

pprint(resource_index)
print(
    "Backend resources:",
    backend_resources,
)

assert backend_resources == [
    "Service/users",
    "Deployment/users",
]


# Problem 39 — Feature Flag Evaluation

A feature is active only when:

- the feature exists
- `enabled` is true
- the current environment appears in `environments`


### Solution 39


In [ ]:
feature_yaml = """
features:
  new_checkout:
    enabled: true
    environments:
      - staging
      - production

  debug_toolbar:
    enabled: true
    environments:
      - development

  old_search:
    enabled: false
    environments:
      - production
"""


def is_feature_enabled(
    config: dict,
    feature_name: str,
    environment: str,
) -> bool:
    features = config.get(
        "features",
        {},
    )

    feature = features.get(
        feature_name
    )

    if feature is None:
        return False

    if not isinstance(feature, dict):
        raise ValueError(
            f"Feature {feature_name!r} "
            "must be a mapping"
        )

    enabled = feature.get(
        "enabled",
        False,
    )

    environments = feature.get(
        "environments",
        [],
    )

    if not isinstance(enabled, bool):
        raise ValueError(
            f"Feature {feature_name!r}: "
            "enabled must be boolean"
        )

    if (
        not isinstance(
            environments,
            list,
        )
        or not all(
            isinstance(item, str)
            for item in environments
        )
    ):
        raise ValueError(
            f"Feature {feature_name!r}: "
            "environments must be "
            "a list of strings"
        )

    return (
        enabled
        and environment
        in environments
    )


feature_cfg = yaml.safe_load(
    feature_yaml
)

assert is_feature_enabled(
    feature_cfg,
    "new_checkout",
    "production",
)

assert not is_feature_enabled(
    feature_cfg,
    "debug_toolbar",
    "production",
)

assert not is_feature_enabled(
    feature_cfg,
    "missing_flag",
    "production",
)

print(
    "Feature flag evaluation passed."
)


# Problem 40 — Final End-to-End Challenge

Build final application configuration from:

- defaults
- YAML text
- `${ENV}` interpolation
- `APP__...` nested overrides

Then validate selected fields, redact secrets for logging, and serialize deterministically.


In [ ]:
final_defaults = {
    "app": {
        "name": "demo",
        "workers": 2,
        "debug": False,
    },
    "database": {
        "host": "localhost",
        "port": 5432,
        "password": "",
    },
    "logging": {
        "level": "INFO",
        "json": False,
    },
}

final_yaml = """
app:
  name: "${APP_NAME:-catalog}"
  workers: 4

database:
  host: "${DB_HOST:-localhost}"
  password: "${DB_PASSWORD}"

logging:
  json: true
"""


### Solution 40


In [ ]:
manager = YamlConfigManager(
    final_defaults
)

config = manager.loads(
    final_yaml,
    env={
        "APP_NAME": "catalog-api",
        "DB_HOST": "db.prod.internal",
        "DB_PASSWORD": (
            "highly-sensitive-value"
        ),
    },
)

config = (
    manager.apply_env_overrides(
        config,
        {
            "APP__APP__WORKERS": "12",
            "APP__LOGGING__LEVEL": (
                "WARNING"
            ),
        },
    )
)


def validate_final_config(
    config: dict,
) -> None:
    app = config.get("app")
    database = config.get(
        "database"
    )
    logging_cfg = config.get(
        "logging"
    )

    if not isinstance(app, dict):
        raise ValueError(
            "'app' must be a mapping"
        )

    if not isinstance(
        database,
        dict,
    ):
        raise ValueError(
            "'database' must be a mapping"
        )

    if not isinstance(
        logging_cfg,
        dict,
    ):
        raise ValueError(
            "'logging' must be a mapping"
        )

    workers = app.get("workers")

    if (
        isinstance(workers, bool)
        or not isinstance(
            workers,
            int,
        )
        or workers <= 0
    ):
        raise ValueError(
            "'app.workers' must be "
            "a positive integer"
        )

    db_port = database.get("port")

    if (
        isinstance(db_port, bool)
        or not isinstance(
            db_port,
            int,
        )
        or not 1 <= db_port <= 65535
    ):
        raise ValueError(
            "'database.port' must "
            "be a valid port"
        )

    level = logging_cfg.get(
        "level"
    )

    if level not in {
        "DEBUG",
        "INFO",
        "WARNING",
        "ERROR",
        "CRITICAL",
    }:
        raise ValueError(
            "'logging.level' is invalid"
        )


validate_final_config(config)

print("Final runtime configuration:")
pprint(config)

print("\nRedacted logging view:")
pprint(
    manager.safe_log_view(config)
)

print(
    "\nDeterministic serialized YAML:"
)
print(manager.dump(config))

assert config["app"]["name"] == (
    "catalog-api"
)
assert config["app"]["workers"] == 12
assert config["database"]["host"] == (
    "db.prod.internal"
)
assert config["logging"]["level"] == (
    "WARNING"
)


# Additional Practice Problems

1. Extend `!duration` to support days.
2. Add a safe `!bytes` tag for `64KiB`, `10MiB`, and `1GiB`.
3. Make `deep_merge` support both list replacement and list appending.
4. Extend `config_diff` to report changed list positions.
5. Add custom line/column formatting to duplicate-key errors.
6. Make `YamlConfigManager` validate a nested schema.
7. Support `${NAME:?custom message}` required-variable syntax.
8. Reject duplicate IDs across a multi-document stream.
9. Add configuration version migration.
10. Reject `debug: true` when `environment: production`.
11. Compute a SHA-256 checksum from deterministic YAML.
12. Serialize `Enum` values safely as strings.
13. Build a safe `!path` tag that normalizes without accessing the filesystem.
14. Validate timeout values against application limits.
15. Extract only an allow-listed subset of configuration for child processes.
16. Validate URL-shaped strings without performing network requests.
17. Add list-of-dictionaries validation with per-item error paths.
18. Implement a configuration deprecation warning system.
19. Add source-priority tracking to show which layer supplied each final value.
20. Build a small test matrix for development, staging, and production configs.


# Best-Practice Checklist

- Prefer `yaml.safe_load` for ordinary and untrusted YAML.
- Prefer `yaml.safe_dump` for plain application data.
- If using `yaml.load`, always supply an intentional restricted loader.
- Never deserialize arbitrary Python objects from untrusted YAML.
- Separate parsing from validation and domain-object construction.
- Reject unknown keys when typos are dangerous.
- Consider rejecting duplicate mapping keys.
- Quote strings that resemble booleans, nulls, numbers, or dates when text is required.
- Validate ranges, types, and cross-field constraints.
- Treat environment variables as untrusted input.
- Never log secrets.
- Prefer deterministic generated YAML for version-control-friendly diffs.
- Use atomic replacement for important configuration writes.
- Consider input-size and nesting limits for untrusted documents.
- Keep custom tags narrow and based on `SafeLoader`.
- Test both valid and invalid configuration paths.
- Remember that YAML parsing is not schema validation.


# Summary

The core engineering pattern is:

```text
YAML text
    -> restricted parsing
    -> plain data
    -> normalization
    -> validation
    -> explicit application objects
```

This is generally safer, easier to test, and easier to maintain than allowing YAML to construct arbitrary Python objects.
